In [1]:
import os
import datetime
import random
import numpy as np
import pandas as pd

from importlib import reload
from openai import OpenAI, RateLimitError
# from any_llm import completion, responses, list_models
from utils import constants, client, metrics, io_results, visualization
from utils.voting import VotingProbabilities, VotingPick, PARTY_NAMES
from utils.models_utils import check_response_api_support
from SoD.data_processing_utils import create_respondent_description


## Experimen Setup

In [2]:
# Loading Data

data = pd.read_csv("data/clean_data.csv", index_col=0)
description_function = create_respondent_description

In [4]:
# Constants

# CLIENT = client.OpenAIClient()
# CLIENT = client.GeminiClient()
# CLIENT = client.AnthropicClient()
CLIENT = client.OpenRouterClient()

# MODEL = constants.GPT_41_NANO
# MODEL = constants.GPT_4o_MINI
# MODEL = constants.GPT_41
# MODEL = constants.GPT_4o
# MODEL = constants.GEMINI_3_FLASH
MODEL = constants.CLAUDE_SONNET_45

TEMPERATURE = 0

VOTING_RESULT = VotingProbabilities
# VOTING_RESULT = VotingPick

# responses_api = check_response_api_support(PROVIDER)

In [5]:
# Prompts

# instructions = """
# Jsi expertní AI asistent specializovaný na analýzu českého politického chování. Tvým úkolem je na základě demografického a postojového profilu respondenta odhadnout jeho volební chování ve volbách do Poslanecké sněmovny Parlamentu ČR v roce 2021.
#
# Tvůj výstup MUSÍ být JSON objekt, který přesně odpovídá Pydantic modelu `VotingResult`. Jiný formát není přípustný.
#
# Dodržuj tato pravidla:
# 1.  Analyzuj VŠECHNY poskytnuté informace o respondentovi (věk, vzdělání, bydliště, příjem, postoje k EU/NATO atd.).
# 2.  Na základě analýzy odhadni dvě klíčové věci:
#     a) Jaká je pravděpodobnost, že respondent vůbec šel k volbám.
#     b) Pokud volil, jaké jsou pravděpodobnosti pro jednotlivé politické strany.
# 3.  Vygeneruj JSON, který bude validní oproti poskytnutým Pydantic modelům.
# 4.  Dbej na to, aby součet pravděpodobností v objektu `voted_or_not` byl přesně 1.0.
# 5.  Dbej na to, aby součet pravděpodobností VŠECH stran v seznamu `parties` byl přesně 1.0."""

prompt_question = " Ve volbách do poslanecké sněmovny v roce 2021 jsem volil:"

In [6]:
if "CLIENT" not in globals():
    CLIENT = client.create_client(MODEL)

## Prompt, Model and Response Showcase

In [7]:
respondent = data.iloc[143]
prompt = description_function(respondent) + prompt_question
prompt

'Jsem muž, je mi 43 let, mé vzdělání je základní + středoškolské vzdělání bez maturity. Žiji v Středočeském kraji, v okresu Beroun a obci o velikosti Méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 40.001 - 60.000 Kč. Nemám ani dobrou, ani špatnou životní úroveň. Spíše se nezajímám o politiku. Ve volbách do poslanecké sněmovny v roce 2021 jsem volil:'

In [8]:
voted = CLIENT.call_llm(MODEL, prompt, VOTING_RESULT, TEMPERATURE)
voted

ValueError: Failed to parse response: {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 64000 tokens, but can only afford 38102. To increase, visit https://openrouter.ai/settings/credits and add more credits', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_2oZH7Uj4J4R9Md30612MKbEFuGV'}

In [ ]:
# Pure GPT call
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
response = client.responses.parse(
    model=MODEL,
    temperature=TEMPERATURE,
    #instructions=INSTRUCTIONS,
    input=prompt,
    text_format=VOTING_RESULT
)

voted = response.output_parsed
response.output_parsed

In [ ]:
# PROVIDER = "ollama"
# MODEL = "jean-luc/tiger-gemma-9b-v3:fp16"

In [ ]:
# # any llm respons - Structure does not work
# response = responses(
#     provider=PROVIDER,
#     model=MODEL,
#     temperature=TEMPERATURE,
#     #instructions=INSTRUCTIONS,
#     input_data=prompt,
#     text=VOTING_RESULT
# )

In [ ]:
# # any llm completion - Only JSON
# response = completion(
#     provider=PROVIDER,
#     model=MODEL,
#     temperature=TEMPERATURE,
#     messages=[
#         # {"role": "system", "content": INSTRUCTIONS},
#         {"role": "user", "content": prompt }
#     ],
#     response_format=VOTING_RESULT
# )
# voted = VotingProbabilities.model_validate_json(response.choices[0].message.content)
# response.choices[0].message

In [ ]:
p = 0
print(voted.voted_or_not)
for party_vote in voted.parties:
    print(f"{party_vote.name}: {party_vote.probability}")
    p += party_vote.probability
print(f"p_sum = {p}")

## Election Simulation Experiment

In [6]:
respondents = data
# respondents = data.sample(n=200, random_state=42)

In [ ]:
voting_results, skipped = client.run_voting_simulation(
    data=respondents,
    prompt_creator= lambda res: description_function(res) + prompt_question,
    response_model=VOTING_RESULT,
    model= MODEL, temperature= TEMPERATURE,
    client= CLIENT
)

print(f"\nProcessed: {len(voting_results)}")
print(f"Failed/Skipped: {len(skipped)}")

Starting processing of 3880 respondents with 10 workers.


  0%|          | 0/3880 [00:00<?, ?it/s]

In [8]:
## Rerun skipped results
while len(skipped)>0:
    skipped_voting_results, skipped = client.run_voting_simulation(
        data=respondents.iloc[skipped],
        prompt_creator= lambda res: description_function(res) + prompt_question,
        response_model=VOTING_RESULT,
        model= MODEL, temperature= TEMPERATURE,
        client= CLIENT
    )
    voting_results.update(skipped_voting_results)

## Save Results

In [8]:
description = io_results.create_parameter_description(CLIENT,MODEL,VOTING_RESULT,len(voting_results),TEMPERATURE)
time = datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S')

In [9]:
io_results.save_results_to_json(voting_results, f"results/voting_results_{description}.json")
io_results.save_results_to_json(voting_results, f"results/voting_results_{description}_{time}.json")

Successfully saved 200 results (with IDs) to results/voting_results_OpenRouterClient_claude-sonnet-4-5_VotingProbabilities_n=200_t=0.0.json
Successfully saved 200 results (with IDs) to results/voting_results_OpenRouterClient_claude-sonnet-4-5_VotingProbabilities_n=200_t=0.0_2026-02-05-20-06-38.json


In [10]:
voting_results_df = io_results.results_to_dataframe(voting_results)
voting_results_df.fillna(0, inplace=True)

In [11]:
voting_results_df.to_csv(f"results/voting_results_{description}.csv", index=True)
voting_results_df.to_csv(f"results/voting_results_{description}_{time}.csv", index=True)

In [12]:
for columns in [constants.ATTENDANCE_COLUMNS,  PARTY_NAMES]:
    row_sums = voting_results_df[columns].sum(axis=1)
    voting_results_df[columns] = voting_results_df[columns].div(row_sums, axis=0)

In [13]:
voting_results_df.to_csv(f"results/normalized_voting_results_{description}.csv", index=True)
voting_results_df.to_csv(f"results/normalized_voting_results_{description}_{time}.csv", index=True)

## Result Evaluation

Validate the quality of the voting results by checking for common issues.

In [14]:
# Evaluate voting results quality
tol = 0.01
bad_voted_sum, bad_party_probs_sum, duplicate_parties = metrics.evaluate_voting_results(voting_results, tol)

# Print details
if bad_voted_sum:
    print(f"[WARN] {len(bad_voted_sum)} respondents where voted+not_voted != 1 (|delta|>{tol}):")
    for rid, v, nv, s in bad_voted_sum[:20]:
        print(f" - ID={rid}: voted={v:.3f}, not_voted={nv:.3f}, sum={s:.3f}")
    if len(bad_voted_sum) > 20:
        print(f" ... and {len(bad_voted_sum) - 20} more")
else:
    print("[OK] All respondents have voted+not_voted summing to 1 within tolerance.")

if bad_party_probs_sum:
    print(f"[WARN] {len(bad_party_probs_sum)} respondents where party probabilities don't sum to 1 (|delta|>{tol}):")
    for rid, s, v in bad_party_probs_sum[:20]:
        print(f" - ID={rid}: voted={v}, prob_sum={s:.3f}")
    if len(bad_party_probs_sum) > 20:
        print(f" ... and {len(bad_party_probs_sum) - 20} more")
else:
    print("[OK] All respondents have party probabilities summing to 1 within tolerance (when parties present).")

if duplicate_parties:
    print(f"[WARN] {len(duplicate_parties)} respondents with duplicate party names:")
    for rid, dups in duplicate_parties[:20]:
        print(f" - ID={rid}: duplicates={dups}")
    if len(duplicate_parties) > 20:
        print(f" ... and {len(duplicate_parties) - 20} more")
else:
    print("[OK] No duplicate party names found in any respondent's parties list.")
# Visualize party probability errors
if bad_party_probs_sum:
    visualization.visualize_party_errors([abs(1 - s) for _, s, _ in bad_party_probs_sum])
else:
    print("No party probability errors to visualize.")

[OK] All respondents have voted+not_voted summing to 1 within tolerance.
[OK] All respondents have party probabilities summing to 1 within tolerance (when parties present).
[WARN] 5 respondents with duplicate party names:
 - ID=3530: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
 - ID=1586: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
 - ID=2227: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
 - ID=3310: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
 - ID=1775: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
No party probability errors to visualize.


In [15]:
# Visualize party probability errors
if bad_party_probs_sum:
    visualization.visualize_party_errors([abs(1-s) for _, s, _ in bad_party_probs_sum])
else:
    print("No party probability errors to visualize.")

No party probability errors to visualize.
